# Residual Learning for Quadrotor Control
## Stage 2: PD Baseline Controller & Disturbance Injection

**Goal of this stage:**
1. Design a cascaded PD controller that stabilizes attitude *and* tracks a 3D position reference
2. Show it works well in clean (undisturbed) conditions
3. Inject realistic disturbances (wind gusts, payload mass uncertainty) and document where and why the classical controller fails

The failure modes we record here become the *training signal* for the residual network in Stage 3.

---
**Run Stage 1 first** (or paste its cells above this notebook) so that `Quadrotor`, `rotation_matrix`, `euler_rate_matrix`, `PARAMS`, `omega_hover`, and `run_simulation` are defined.

## 1. Imports & Carry-Forward from Stage 1

Everything below re-defines Stage 1 so this notebook is **self-contained**.
If you already ran Stage 1 in the same session, these cells are harmless to re-run.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Line3DCollection
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)

# ── Physical constants ───────────────────────────────────────────────────────
g = 9.81

PARAMS = {
    'm'       : 0.5,
    'I'       : np.diag([4.9e-3, 4.9e-3, 8.8e-3]),
    'l'       : 0.175,
    'kf'      : 2.0e-6,
    'km'      : 5.0e-9,
    'max_rpm' : 8000.0,
    'g'       : g,
}

omega_hover = np.sqrt(PARAMS['m'] * g / (4 * PARAMS['kf']))

# ── Rotation utilities ───────────────────────────────────────────────────────
def rotation_matrix(phi, theta, psi):
    cphi,sphi = np.cos(phi),np.sin(phi)
    cth, sth  = np.cos(theta),np.sin(theta)
    cpsi,spsi = np.cos(psi),np.sin(psi)
    Rx = np.array([[1,0,0],[0,cphi,-sphi],[0,sphi,cphi]])
    Ry = np.array([[cth,0,sth],[0,1,0],[-sth,0,cth]])
    Rz = np.array([[cpsi,-spsi,0],[spsi,cpsi,0],[0,0,1]])
    return Rz @ Ry @ Rx

def euler_rate_matrix(phi, theta):
    cphi,sphi = np.cos(phi),np.sin(phi)
    cth = np.cos(theta); tth = np.tan(theta)
    return np.array([[1, sphi*tth, cphi*tth],
                     [0, cphi,    -sphi    ],
                     [0, sphi/cth, cphi/cth]])

# ── Quadrotor simulator ──────────────────────────────────────────────────────
class Quadrotor:
    def __init__(self, params):
        self.m        = params['m']
        self.I        = params['I']
        self.Iinv     = np.linalg.inv(self.I)
        self.l        = params['l']
        self.kf       = params['kf']
        self.km       = params['km']
        self.g        = params['g']
        self.max_omega = params['max_rpm'] * 2*np.pi/60
        self.spin_dirs = np.array([1,-1,1,-1], dtype=float)

    def reset(self, state=None):
        if state is not None:
            self.state = state.copy()
        else:
            self.state = np.zeros(12)
            self.state[2] = 1.0
        return self.state.copy()

    def _forces_torques(self, omega_rotors):
        omega    = np.clip(omega_rotors, 0, self.max_omega)
        omega_sq = omega**2
        F_rotors = self.kf * omega_sq
        F_total  = np.sum(F_rotors)
        tau_phi   = self.l * (F_rotors[3] - F_rotors[1])
        tau_theta = self.l * (F_rotors[0] - F_rotors[2])
        tau_psi   = self.km * np.sum(self.spin_dirs * omega_sq)
        return F_total, np.array([tau_phi, tau_theta, tau_psi])

    def derivatives(self, state, omega_rotors, disturbance=None):
        pos,vel,angles,omega = state[0:3],state[3:6],state[6:9],state[9:12]
        phi,theta,psi = angles
        R = rotation_matrix(phi, theta, psi)
        F_total, tau = self._forces_torques(omega_rotors)
        thrust_world = R @ np.array([0,0,F_total])
        gravity      = np.array([0,0,-self.m*self.g])
        dist         = disturbance if disturbance is not None else np.zeros(3)
        acc          = (thrust_world + gravity + dist) / self.m
        omega_dot    = self.Iinv @ (tau - np.cross(omega, self.I @ omega))
        euler_dot    = euler_rate_matrix(phi, theta) @ omega
        dstate = np.zeros(12)
        dstate[0:3]=vel; dstate[3:6]=acc; dstate[6:9]=euler_dot; dstate[9:12]=omega_dot
        return dstate

    def step(self, omega_rotors, dt, disturbance=None):
        x  = self.state
        k1 = self.derivatives(x,            omega_rotors, disturbance)
        k2 = self.derivatives(x+dt/2*k1,    omega_rotors, disturbance)
        k3 = self.derivatives(x+dt/2*k2,    omega_rotors, disturbance)
        k4 = self.derivatives(x+dt*k3,      omega_rotors, disturbance)
        self.state = x + (dt/6)*(k1+2*k2+2*k3+k4)
        self.state[8] = (self.state[8]+np.pi)%(2*np.pi)-np.pi
        return self.state.copy()

print("Stage 1 definitions loaded.")
print(f"Hover omega: {omega_hover:.1f} rad/s")

## 2. Cascaded PD Controller Design

We use a standard two-loop cascaded architecture — the same structure used in real flight controllers (ArduPilot, PX4):

```
 Position ref ──► [Outer PD] ──► Attitude ref ──► [Inner PD] ──► Rotor speeds
      ▲                ▲                ▲               ▲
   x,y,z            xdot           phi,theta        p,q,r
```

**Outer loop (position):** Computes the desired roll/pitch angles to accelerate toward the target position.

**Inner loop (attitude):** Computes differential rotor torques to drive roll/pitch/yaw to their desired values.

**Altitude:** Controlled separately via total thrust — a PD on the z-error.

### Control allocation
The mixer maps desired total thrust $F$ and torques $[\tau_\phi, \tau_\theta, \tau_\psi]$ to individual rotor speeds:

$$\begin{bmatrix} F \\ \tau_\phi \\ \tau_\theta \\ \tau_\psi \end{bmatrix}
= \underbrace{\begin{bmatrix}
k_f & k_f & k_f & k_f \\
0 & -k_f l & 0 & k_f l \\
k_f l & 0 & -k_f l & 0 \\
k_m & -k_m & k_m & -k_m
\end{bmatrix}}_{\text{mixer matrix } M}
\begin{bmatrix} \omega_1^2 \\ \omega_2^2 \\ \omega_3^2 \\ \omega_4^2 \end{bmatrix}$$

We invert $M$ to solve for $\omega_i^2$ from desired wrench.

In [ ]:
class PDController:
    """
    Cascaded PD controller for quadrotor position + attitude stabilization.

    Architecture:
      Outer loop: position error  -> desired roll/pitch angles
      Altitude:   z error         -> total thrust
      Inner loop: attitude error  -> differential torques
      Mixer:      wrench          -> rotor omega commands

    All gains are hand-tuned for the 500g, 175mm-arm quadrotor in PARAMS.
    """

    def __init__(self, params, gains=None):
        self.m   = params['m']
        self.g   = params['g']
        self.kf  = params['kf']
        self.km  = params['km']
        self.l   = params['l']
        self.max_omega = params['max_rpm'] * 2*np.pi/60

        # ── Gains ────────────────────────────────────────────────────────────
        # Position outer loop (produces desired angles)
        # Units: [N / m] and [N / (m/s)]
        self.Kp_pos = np.array([1.2, 1.2, 4.0])   # P gain  [x, y, z]
        self.Kd_pos = np.array([0.8, 0.8, 2.5])   # D gain  [x, y, z]

        # Attitude inner loop (produces torques)
        # Units: [N·m / rad] and [N·m / (rad/s)]
        self.Kp_att = np.array([0.6,  0.6,  0.3 ])  # Note to self- Adjusted wrights for better tracking
        self.Kd_att = np.array([0.15, 0.15, 0.08])

        # Allow custom gains (used for sensitivity analysis)
        if gains is not None:
            for k, v in gains.items():
                setattr(self, k, v)

        # ── Mixer matrix & its inverse ───────────────────────────────────────
        kf, km, l = self.kf, self.km, self.l
        self.M = np.array([
            [ kf,    kf,    kf,    kf   ],   # total thrust
            [ 0,    -kf*l,  0,     kf*l ],   # roll torque  (rotors 4,2)
            [ kf*l,  0,    -kf*l,  0    ],   # pitch torque (rotors 1,3)
            [ km,   -km,    km,   -km   ],   # yaw torque   (spin directions)
        ])
        self.Minv = np.linalg.inv(self.M)

        # Max thrust that can be generated
        self.F_max = 4 * self.kf * self.max_omega**2

    def compute(self, state, ref_pos, ref_vel=None, ref_yaw=0.0):
        """
        Compute rotor speed commands.

        Args:
            state   : (12,) quadrotor state
            ref_pos : (3,)  desired [x, y, z] position
            ref_vel : (3,)  desired [xdot, ydot, zdot] (feed-forward), default zero
            ref_yaw : float desired yaw angle (rad)

        Returns:
            omega_cmd : (4,) rotor speed commands in rad/s
            debug     : dict with internal signals for analysis
        """
        if ref_vel is None:
            ref_vel = np.zeros(3)

        pos    = state[0:3]
        vel    = state[3:6]
        angles = state[6:9]   # phi, theta, psi
        omega  = state[9:12]  # p, q, r
        phi, theta, psi = angles

        # ── Outer loop: position PD ──────────────────────────────────────────
        pos_err = ref_pos - pos
        vel_err = ref_vel - vel

        # Desired acceleration in world frame
        acc_des = self.Kp_pos * pos_err + self.Kd_pos * vel_err

        # ── Altitude: map z-acceleration to total thrust ─────────────────────
        # F/m = az + g  =>  F = m*(az_des + g)
        F_des = self.m * (acc_des[2] + self.g)
        F_des = np.clip(F_des, 0.1, self.F_max)

        # ── Outer loop: desired attitude from horizontal acceleration ─────────
        # From quadrotor kinematics (small angle approximation not required here):
        # ax = (F/m)*(sin(psi)*phi + cos(psi)*theta)
        # ay = (F/m)*(-cos(psi)*phi + sin(psi)*theta)
        # Solving for desired phi, theta:
        ax_des = acc_des[0]
        ay_des = acc_des[1]
        cpsi, spsi = np.cos(psi), np.sin(psi)

        # Desired tilt angles (clipped to ±30°)
        theta_des =  (ax_des * cpsi + ay_des * spsi) * self.m / F_des
        phi_des   =  -1 * (ax_des * spsi - ay_des * cpsi) * self.m / F_des
        theta_des = np.clip(theta_des, -np.radians(30), np.radians(30))
        phi_des   = np.clip(phi_des,   -np.radians(30), np.radians(30))
        psi_des   = ref_yaw

        # ── Inner loop: attitude PD ──────────────────────────────────────────
        att_err = np.array([phi_des - phi,
                            theta_des - theta,
                            psi_des   - psi])
        # Wrap yaw error to [-pi, pi]
        att_err[2] = (att_err[2] + np.pi) % (2*np.pi) - np.pi

        # Angular velocity error (desired body rates = 0 for set-point tracking)
        omega_err = -omega  # we want omega -> 0

        # Desired torques
        tau_des = self.Kp_att * att_err + self.Kd_att * omega_err

        # ── Control allocation: wrench -> rotor omega^2 ──────────────────────
        wrench   = np.array([F_des, tau_des[0], tau_des[1], tau_des[2]])
        omega_sq = self.Minv @ wrench

        # Physical constraints: omega^2 must be >= 0
        omega_sq = np.clip(omega_sq, 0, self.max_omega**2)
        omega_cmd = np.sqrt(omega_sq)

        debug = {
            'pos_err'  : pos_err,
            'att_err'  : att_err,
            'att_des'  : np.array([phi_des, theta_des, psi_des]),
            'F_des'    : F_des,
            'tau_des'  : tau_des,
        }

        return omega_cmd, debug


# Instantiate
controller = PDController(PARAMS)
print("PDController instantiated.")
print(f"Mixer matrix condition number: {np.linalg.cond(controller.M):.1f}  (well-conditioned < 1000)")

## 3. Trajectory Generator

We define three reference trajectories to test the controller:

1. **Hover** — static setpoint at [0, 0, 2m]
2. **Step response** — sudden 3m position step in x
3. **Lemniscate (figure-8)** — smooth, time-varying 3D path

The lemniscate is a demanding test because it requires continuously varying roll and pitch while maintaining altitude — it will expose disturbance sensitivity clearly.

In [ ]:
class TrajectoryGenerator:
    """Reference trajectory generator. Returns (pos, vel) at time t."""

    @staticmethod
    def hover(t, pos=np.array([0., 0., 2.])):
        """Static hover at a fixed position."""
        return pos.copy(), np.zeros(3)

    @staticmethod
    def step(t, step_time=1.0, step_pos=np.array([3., 0., 2.])):
        """Step from origin to step_pos at t=step_time."""
        if t < step_time:
            return np.array([0., 0., 2.]), np.zeros(3)
        else:
            return step_pos.copy(), np.zeros(3)

    @staticmethod
    def lemniscate(t, scale=2.0, period=8.0, z=2.0):
        """
        Bernoulli lemniscate (figure-8) in the x-y plane at constant altitude.
        Parametric form:
            x(t) =  a * cos(w*t) / (1 + sin²(w*t))
            y(t) =  a * sin(w*t)*cos(w*t) / (1 + sin²(w*t))
        Velocity computed analytically.
        """
        a = scale
        w = 2*np.pi / period
        s, c = np.sin(w*t), np.cos(w*t)
        denom = 1 + s**2

        x =  a * c / denom
        y =  a * s * c / denom

        # Analytical derivatives
        ddenom = 2*s*c*w
        xdot = (-a*s*w*denom - a*c*ddenom) / denom**2
        ydot = ( a*(c**2 - s**2)*w*denom - a*s*c*ddenom) / denom**2

        return np.array([x, y, z]), np.array([xdot, ydot, 0.])


traj = TrajectoryGenerator()

# Preview the trajectories
t_preview = np.linspace(0, 8, 500)
lemni_pos = np.array([traj.lemniscate(t)[0] for t in t_preview])

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(lemni_pos[:,0], lemni_pos[:,1], color='#457b9d', lw=2.5)
ax.scatter([0],[0], s=80, color='#e63946', zorder=5, label='Start')
ax.set_xlabel('X (m)'); ax.set_ylabel('Y (m)')
ax.set_title('Lemniscate Reference Path (top-down view)', fontweight='bold')
ax.set_aspect('equal'); ax.grid(True, alpha=0.3); ax.legend()
plt.tight_layout(); plt.savefig('reference_path.png', dpi=150, bbox_inches='tight')
plt.show()
print("Trajectory generator ready.")

## 4. Simulation Runner with Data Logging

We extend the Stage 1 runner to log controller internals — error signals, desired attitudes, thrust commands — which we will need for training the residual network.

In [ ]:
def run_controlled(
    quad, controller, traj_fn,
    t_end=10.0, dt=0.002,
    disturbance_fn=None,
    mass_factor=1.0,        # multiplicative mass perturbation
    verbose=True
):
    """
    Run a full controlled simulation and return a rich data log.

    Args:
        quad          : Quadrotor instance (will be reset internally)
        controller    : PDController instance
        traj_fn       : callable(t) -> (pos_ref, vel_ref)
        t_end         : simulation duration (s)
        dt            : timestep (s)
        disturbance_fn: callable(t) -> (3,) force vector, or None
        mass_factor   : simulates payload (e.g. 1.3 = 30% heavier than controller expects)
        verbose       : print summary stats

    Returns:
        log: dict with arrays of shape (N,) or (N,3) / (N,12)
    """
    t_span = np.arange(0, t_end, dt)
    N = len(t_span)

    # Storage
    log = {
        't'         : t_span,
        'state'     : np.zeros((N, 12)),
        'ref_pos'   : np.zeros((N, 3)),
        'pos_err'   : np.zeros((N, 3)),
        'att_err'   : np.zeros((N, 3)),
        'att_des'   : np.zeros((N, 3)),
        'omega_cmd' : np.zeros((N, 4)),
        'F_des'     : np.zeros(N),
        'disturbance': np.zeros((N, 3)),
    }

    # Start 0.5m below hover target to see transient
    init = np.zeros(12)
    init[2] = 1.5
    quad.reset(init)

    # Temporarily perturb quad mass to simulate payload uncertainty
    true_mass = quad.m
    quad.m = true_mass * mass_factor   # physics sees heavier drone
    # Controller still uses nominal mass — this is the mismatch!

    for i, t in enumerate(t_span):
        ref_pos, ref_vel = traj_fn(t)
        dist = disturbance_fn(t) if disturbance_fn else None

        # Controller computes with *nominal* model (doesn't know true mass)
        omega_cmd, dbg = controller.compute(quad.state, ref_pos, ref_vel)

        # Log
        log['state'][i]      = quad.state
        log['ref_pos'][i]    = ref_pos
        log['pos_err'][i]    = dbg['pos_err']
        log['att_err'][i]    = dbg['att_err']
        log['att_des'][i]    = dbg['att_des']
        log['omega_cmd'][i]  = omega_cmd
        log['F_des'][i]      = dbg['F_des']
        log['disturbance'][i]= dist if dist is not None else np.zeros(3)

        # Step physics (true mass, with disturbance)
        quad.step(omega_cmd, dt, disturbance=dist)

        if quad.state[2] < -0.5:
            if verbose: print(f"  [!] Drone crashed at t={t:.2f}s")
            log = {k: v[:i+1] for k, v in log.items()}
            break

    # Restore true mass
    quad.m = true_mass

    # Compute scalar position error norm
    log['pos_err_norm'] = np.linalg.norm(log['pos_err'], axis=1)

    if verbose:
        steady = log['pos_err_norm'][len(t_span)//2:]  # second half
        print(f"  Steady-state RMS position error : {np.sqrt(np.mean(steady**2)):.4f} m")
        print(f"  Peak position error             : {log['pos_err_norm'].max():.4f} m")

    return log

print("Simulation runner ready.")

## 5. Clean Baseline: Controller Without Disturbance

First we confirm the PD controller works well in nominal conditions on all three trajectories.

In [ ]:
quad       = Quadrotor(PARAMS)
controller = PDController(PARAMS)

# ── Run all three trajectories, no disturbance ───────────────────────────────
print("=== HOVER ===")
log_hover = run_controlled(quad, controller,
                           traj_fn=lambda t: traj.hover(t),
                           t_end=6.0)

print("\n=== STEP ===")
log_step = run_controlled(quad, controller,
                          traj_fn=lambda t: traj.step(t),
                          t_end=8.0)

print("\n=== LEMNISCATE ===")
log_lemni = run_controlled(quad, controller,
                           traj_fn=lambda t: traj.lemniscate(t),
                           t_end=16.0)

# ── Plot: position tracking for all three ────────────────────────────────────
fig, axes = plt.subplots(3, 3, figsize=(16, 10))
fig.suptitle('Baseline PD Controller — Clean Conditions (No Disturbance)',
             fontsize=13, fontweight='bold')

logs   = [log_hover, log_step, log_lemni]
titles = ['Hover', 'Step Response', 'Lemniscate']
axes_labels = ['X (m)', 'Y (m)', 'Z (m)']
colors_ref  = ['#a8dadc', '#a8dadc', '#a8dadc']
colors_act  = ['#457b9d', '#e63946', '#2a9d8f']

for row, (log, title, col) in enumerate(zip(logs, titles, colors_act)):
    t = log['t']
    for col_idx in range(3):
        ax = axes[row, col_idx]
        ax.plot(t, log['ref_pos'][:, col_idx], '--',
                color='gray', lw=1.5, label='Reference', alpha=0.8)
        ax.plot(t, log['state'][:, col_idx],
                color=col, lw=2, label='Actual')
        ax.set_xlabel('Time (s)')
        ax.set_ylabel(axes_labels[col_idx])
        if col_idx == 0:
            ax.set_ylabel(f'{title}\n{axes_labels[col_idx]}')
        ax.grid(True, alpha=0.3)
        if row == 0 and col_idx == 0:
            ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('clean_baseline.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Disturbance Models

We design three physically motivated disturbances:

| Disturbance | Physical meaning | Form |
|---|---|---|
| **Sustained wind** | Constant headwind (e.g. 3 m/s breeze) | Constant force in x |
| **Wind gusts** | Turbulent, bursty wind events | Random impulses at irregular intervals |
| **Payload mass** | Picking up a package mid-flight | Step in mass at t=4s |

These represent real-world failure modes for classical controllers — the PD was designed for a specific model and cannot compensate for unknown additive forces or model mismatch.

In [ ]:
def make_sustained_wind(force_N=1.5, direction=0):
    """
    Constant wind force.
    force_N: magnitude in Newtons (1.5N ≈ 3 m/s wind on 0.5kg drone)
    direction: angle in x-y plane (radians)
    """
    fx = force_N * np.cos(direction)
    fy = force_N * np.sin(direction)
    force = np.array([fx, fy, 0.0])
    return lambda t: force


def make_wind_gusts(peak_N=3.0, gust_duration=0.3, rate=0.5, seed=7):
    """
    Stochastic gusts: random direction, random timing.
    peak_N       : peak gust force (Newtons)
    gust_duration: how long each gust lasts (s)
    rate         : average gusts per second
    """
    rng = np.random.default_rng(seed)
    # Pre-generate gust events as (start_time, force_vector)
    t_max = 30.0
    gusts = []
    t_cur = 0.0
    while t_cur < t_max:
        gap       = rng.exponential(1.0 / rate)
        t_start   = t_cur + gap
        angle     = rng.uniform(0, 2*np.pi)
        magnitude = rng.uniform(0.5*peak_N, peak_N)
        fvec      = magnitude * np.array([np.cos(angle), np.sin(angle), 0.0])
        gusts.append((t_start, t_start + gust_duration, fvec))
        t_cur = t_start + gust_duration

    def disturbance(t):
        for t_start, t_end, fvec in gusts:
            if t_start <= t < t_end:
                # Smooth ramp-up / ramp-down using sin envelope
                phase = (t - t_start) / gust_duration * np.pi
                return fvec * np.sin(phase)
        return np.zeros(3)

    return disturbance


# Preview disturbances over time
t_prev = np.linspace(0, 10, 2000)
wind_fn  = make_sustained_wind(force_N=1.5)
gusts_fn = make_wind_gusts(peak_N=3.0)
wind_vals  = np.array([wind_fn(t) for t in t_prev])
gusts_vals = np.array([gusts_fn(t) for t in t_prev])

fig, axes = plt.subplots(2, 1, figsize=(12, 6))
fig.suptitle('Disturbance Profiles', fontsize=13, fontweight='bold')

axes[0].plot(t_prev, wind_vals[:,0], color='#457b9d', lw=2, label='Fx')
axes[0].plot(t_prev, wind_vals[:,1], color='#e63946', lw=2, label='Fy')
axes[0].set_title('Sustained Wind (1.5 N)')
axes[0].set_ylabel('Force (N)'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(t_prev, gusts_vals[:,0], color='#457b9d', lw=1.5, label='Fx')
axes[1].plot(t_prev, gusts_vals[:,1], color='#e63946', lw=1.5, label='Fy')
axes[1].set_title('Stochastic Wind Gusts (peak 3 N)')
axes[1].set_xlabel('Time (s)'); axes[1].set_ylabel('Force (N)')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('disturbance_profiles.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Controller Failure Analysis

We now run the **lemniscate** trajectory under each disturbance type. This is the most demanding scenario — it requires constant attitude changes, so disturbances compound rather than being averaged out.

The quantitative result — how much position error increases — is what justifies needing a residual compensator.

In [ ]:
traj_fn = lambda t: traj.lemniscate(t, period=8.0)

print("Running disturbance scenarios on lemniscate trajectory...\n")

print("[1/4] Clean baseline")
log_clean   = run_controlled(quad, controller, traj_fn, t_end=16.0)

print("\n[2/4] Sustained wind (1.5 N)")
log_wind    = run_controlled(quad, controller, traj_fn, t_end=16.0,
                             disturbance_fn=make_sustained_wind(1.5))

print("\n[3/4] Stochastic gusts (peak 3 N)")
log_gusts   = run_controlled(quad, controller, traj_fn, t_end=16.0,
                             disturbance_fn=make_wind_gusts(3.0))

print("\n[4/4] Payload mass uncertainty (+30%)")
log_payload = run_controlled(quad, controller, traj_fn, t_end=16.0,
                             mass_factor=1.30)

In [ ]:
# ── Figure 1: Position error over time ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('PD Controller: Clean vs Disturbed — Lemniscate Trajectory',
             fontsize=13, fontweight='bold')

scenarios = [
    (log_clean,   '#2a9d8f', 'Clean baseline'),
    (log_wind,    '#457b9d', 'Sustained wind (1.5 N)'),
    (log_gusts,   '#e63946', 'Stochastic gusts (peak 3 N)'),
    (log_payload, '#f4a261', 'Payload +30% mass'),
]

ax = axes[0]
for log, col, label in scenarios:
    ax.plot(log['t'], log['pos_err_norm'], color=col, lw=2, label=label, alpha=0.9)
ax.set_xlabel('Time (s)'); ax.set_ylabel('Position Error |e| (m)')
ax.set_title('Position Error Magnitude')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
ax.set_ylim(bottom=0)

# ── Figure 2: 3D trajectories ─────────────────────────────────────────────────
ax3d = fig.add_subplot(122, projection='3d')
for log, col, label in scenarios:
    s = log['state']
    ax3d.plot(s[:,0], s[:,1], s[:,2], color=col, lw=1.8, label=label, alpha=0.85)

# Reference path
ref_xyz = log_clean['ref_pos']
ax3d.plot(ref_xyz[:,0], ref_xyz[:,1], ref_xyz[:,2], 'k--',
          lw=1, alpha=0.4, label='Reference')

ax3d.set_xlabel('X (m)'); ax3d.set_ylabel('Y (m)'); ax3d.set_zlabel('Z (m)')
ax3d.set_title('3D Actual Trajectories')
ax3d.legend(fontsize=8, loc='upper left')

plt.tight_layout()
plt.savefig('failure_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Summary table ─────────────────────────────────────────────────────────────
print("\n{'='*55}")
print(f"{'Scenario':<30} {'Steady RMS (m)':>14} {'Peak err (m)':>13}")
print('='*55)
for log, _, label in scenarios:
    steady = log['pos_err_norm'][len(log['t'])//2:]
    rms    = np.sqrt(np.mean(steady**2))
    peak   = log['pos_err_norm'].max()
    print(f"{label:<30} {rms:>14.4f} {peak:>13.4f}")
print('='*55)

## 8. Data Collection for Stage 3

We now generate a **training dataset** by running the PD controller under many different disturbance conditions.

**What we store per timestep:**
- State: position, velocity, attitude, angular rate → input features for the residual network
- Controller output: rotor commands
- **Residual signal**: the force that the PD controller *failed to compensate* — computed as the difference between actual and desired acceleration

This residual is exactly what the neural network will learn to predict and correct.

In [ ]:
def collect_training_data(quad, controller, n_episodes=40, t_end=16.0, dt=0.002):
    """
    Collect (state, controller_output, residual_force) tuples across
    diverse disturbance conditions.

    The residual force is computed as:
        f_residual = m * (a_actual - a_commanded)
    where a_actual is derived from finite-differencing velocity,
    and a_commanded comes from the PD thrust command.

    Returns:
        X: (N_total, 12) input features (state)
        Y: (N_total, 3)  target residual force (world frame, Newtons)
    """
    rng = np.random.default_rng(2024)
    all_X, all_Y = [], []

    trajectories = [
        lambda t: traj.hover(t),
        lambda t: traj.lemniscate(t, period=6.0),
        lambda t: traj.lemniscate(t, period=10.0),
        lambda t: traj.step(t, step_pos=np.array([2., 1., 2.5])),
    ]

    for ep in range(n_episodes):
        # Randomize disturbance type per episode
        disturbance_type = rng.choice(['wind', 'gusts', 'none'])
        if disturbance_type == 'wind':
            force_mag = rng.uniform(0.5, 2.5)
            direction = rng.uniform(0, 2*np.pi)
            dist_fn   = make_sustained_wind(force_mag, direction)
        elif disturbance_type == 'gusts':
            peak = rng.uniform(1.0, 4.0)
            dist_fn = make_wind_gusts(peak, seed=int(rng.integers(0, 9999)))
        else:
            dist_fn = None

        # Randomize mass factor
        mass_factor = rng.uniform(0.85, 1.35)

        # Pick random trajectory
        traj_fn = trajectories[ep % len(trajectories)]

        # Run episode
        log = run_controlled(
            quad, controller, traj_fn,
            t_end=t_end, dt=dt,
            disturbance_fn=dist_fn,
            mass_factor=mass_factor,
            verbose=False
        )

        # ── Compute residual force ────────────────────────────────────────
        # Actual acceleration from finite difference of velocity
        vel  = log['state'][:, 3:6]
        a_actual = np.gradient(vel, dt, axis=0)  # numerical derivative

        # Commanded acceleration from thrust (world frame, gravity already included)
        # F_des / m gives desired az;  actual ax,ay assumed 0 from commanded attitude
        # We compute commanded net force from actual rotor commands
        a_cmd = np.zeros_like(a_actual)
        for i in range(len(log['t'])):
            state_i   = log['state'][i]
            phi, theta, psi = state_i[6:9]
            R         = rotation_matrix(phi, theta, psi)
            F_des_i   = log['F_des'][i]
            thrust_w  = R @ np.array([0, 0, F_des_i])
            gravity_w = np.array([0, 0, -quad.m * quad.g])
            a_cmd[i]  = (thrust_w + gravity_w) / quad.m

        # Residual = what the controller couldn't account for
        f_residual = quad.m * (a_actual - a_cmd)   # shape (N, 3), Newtons

        # Skip first and last few samples (finite-diff boundary effects)
        skip = 5
        all_X.append(log['state'][skip:-skip])
        all_Y.append(f_residual[skip:-skip])

        if (ep+1) % 10 == 0:
            print(f"  Episode {ep+1}/{n_episodes} — "
                  f"disturbance: {disturbance_type}, mass: {mass_factor:.2f}x")

    X = np.concatenate(all_X, axis=0)
    Y = np.concatenate(all_Y, axis=0)
    return X, Y


print("Collecting training data (40 episodes × 16s)...")
X_train, Y_train = collect_training_data(quad, controller, n_episodes=40)
print(f"\nDataset shape: X={X_train.shape}, Y={Y_train.shape}")
print(f"Total samples : {len(X_train):,}")
print(f"Residual force stats (N):")
print(f"  Mean magnitude : {np.linalg.norm(Y_train, axis=1).mean():.4f}")
print(f"  Max  magnitude : {np.linalg.norm(Y_train, axis=1).max():.4f}")

# Save dataset for Stage 3
np.save('X_train.npy', X_train)
np.save('Y_train.npy', Y_train)
print("\nDataset saved as X_train.npy and Y_train.npy")

In [ ]:
# ── Visualize residual force distribution ─────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Training Data: Residual Force Distribution', fontsize=13, fontweight='bold')

labels = ['Fx residual (N)', 'Fy residual (N)', 'Fz residual (N)']
colors = ['#457b9d', '#e63946', '#2a9d8f']

for i, (ax, label, col) in enumerate(zip(axes, labels, colors)):
    ax.hist(Y_train[:, i], bins=80, color=col, alpha=0.8, edgecolor='none')
    ax.axvline(0, color='k', lw=1.5, ls='--')
    ax.set_xlabel(label)
    ax.set_ylabel('Count')
    ax.set_title(f'μ={Y_train[:,i].mean():.3f}, σ={Y_train[:,i].std():.3f}')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('residual_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print("Non-zero residual forces confirm disturbances were captured.")
print("The asymmetric Fz distribution reflects mass perturbations (heavier drone needs more thrust).")

]